# 31 — Parsing Error Handling
**Goal:** Gracefully handle corrupt files, empty documents, and edge cases.

## 1. Common Failure Modes

In [ ]:
faults = [
    "Corrupt PDF/DOCX (broken ZIP structure)",
    "Empty files (0 bytes)",
    "Password-protected documents",
    "File with only images (no text layer)",
    "Wrong extension (.pdf but is .docx)",
    "Truncated downloads",
    "Encoding errors (non-UTF8 content)",
]
for f in faults: print(f"  - {f}")

## 2. Magic Byte File Detection

In [ ]:
def detect_type(filepath):
    sigs = {b'%PDF': 'pdf', b'PK\x03\x04': 'docx', b'\x89PNG': 'png', b'\xff\xd8\xff': 'jpg'}
    with open(filepath, 'rb') as f:
        h = f.read(4)
    for sig, t in sigs.items():
        if h.startswith(sig): return t
    return 'unknown'

import tempfile, os
with tempfile.NamedTemporaryFile(suffix='.pdf', delete=False) as f:
    f.write(b'%PDF-1.4 fake')
    fn = f.name
print(f"File: {fn}")
print(f"Detected: {detect_type(fn)} (vs extension: .pdf)")
os.unlink(fn)

## 3. Robust Document Parser with Fallbacks

In [ ]:
class SafeParser:
    def __init__(self):
        self.strategies = []
    def add(self, name, fn):
        self.strategies.append((name, fn))
    def parse(self, filepath):
        for name, fn in self.strategies:
            try:
                text = fn(filepath)
                if text and text.strip():
                    return text
            except Exception as e:
                print(f"  {name} failed: {e}")
        return ""

p = SafeParser()
p.add("pdfplumber", lambda f: "extracted pdf text")
p.add("PyMuPDF", lambda f: "text from pymupdf")
p.add("OCR", lambda f: "text from ocr")
print(f"Result: '{p.parse('test.pdf')}'")

## Summary: Always validate file types by magic bytes, not extension. Layer fallbacks.